# Proyecto EDA: Análisis exploratorio de datos COVID-19 en Estados Unidos

## 1. Introducción

El objetivo de este proyecto es realizar un análisis exploratorio de datos (EDA) sobre la evolución de la pandemia de COVID-19 en Estados Unidos.

A través de Python y sus principales librerías de análisis de datos, se estudiará la evolución de los casos confirmados, las pruebas realizadas y los fallecimientos registrados en los diferentes estados.

El análisis permitirá identificar tendencias temporales, diferencias entre estados y posibles patrones relacionados con el impacto de la pandemia.

## 2. Objetivos

- Analizar la estructura y calidad del conjunto de datos.
- Identificar y tratar valores faltantes, duplicados e inconsistencias.
- Estudiar la evolución temporal de los principales indicadores.
- Comparar el impacto de la pandemia entre estados.
- Crear visualizaciones que faciliten la interpretación de los resultados.
- Comunicar los principales hallazgos.

### 1.1. Preguntas de investigación

Antes de analizar los datos nos planteamos estas tres preguntas:

1. **Evolución temporal:** ¿cómo evolucionaron los casos nuevos y los fallecimientos durante el periodo?
2. **Comparación entre estados:** ¿qué estados concentraron más casos positivos?
3. **Relación pruebas–casos:** ¿existe relación entre el número de pruebas realizadas y los casos positivos?

Estas preguntas se responden en la sección 7.

## 2. Cargar los datos

En esta sección descargamos el dataset desde la API pública del *COVID Tracking Project* y lo dejamos preparado para el análisis.

### 2.1. Importar librerías

Primero, importamos las librerías necesarias para la descarga, manipulación y visualización de los datos.

In [ ]:
from pathlib import Path

import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.float_format", lambda x: f"{x:,.0f}")

### 2.2. Descargar o leer el dataset

Para no descargar los datos en cada ejecución, el notebook lee el CSV local si ya existe. Si no existe, descarga los datos desde la API y los guarda en `data/covid_us.csv`.

In [ ]:
URL = "https://api.covidtracking.com/v1/states/daily.json"
DATA_PATH = Path("../data/covid_us.csv")

if DATA_PATH.exists():
    df = pd.read_csv(DATA_PATH)
    print(f"Datos cargados desde {DATA_PATH}")
else:
    response = requests.get(URL, timeout=30)
    response.raise_for_status()
    df = pd.DataFrame(response.json())
    DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(DATA_PATH, index=False)
    print("Datos descargados desde la API")

df.head()

## 3. Revisión de la estructura del dataset

Antes de limpiar los datos, revisamos su tamaño, los tipos de cada columna y las primeras filas para entender qué información contienen.

In [ ]:
# Número de filas y columnas
df.shape

In [ ]:
# Información general
df.info()

In [ ]:
# Primeras filas
df.head()

In [ ]:
# Estadísticas descriptivas
df.describe()

El dataset contiene **20.780 registros y 56 columnas**: un registro por estado o territorio y día, entre el **13 de enero de 2020** y el **7 de marzo de 2021** (56 estados/territorios en total).

`info()` muestra que la mayoría de las columnas son numéricas, pero `date` viene como entero (`20210307`) y hay muchas columnas con gran cantidad de valores nulos. `describe()` confirma que las medidas acumuladas están muy sesgadas: la media de casos positivos es mucho mayor que la mediana, porque unos pocos estados grandes concentran gran parte de los casos.